# QMatBridge functionality tour

A second notebook, complementary to the quickstart. The quickstart looks at the six Tier-1 entries and the fingerprint; this one exercises the rest of the toolbox and draws the figures that go with it:

1. **Registry.** What adapters and exporters are installed.
2. **Structures.** Draw the unit cells that the entries record (schema 0.2 atomic positions).
3. **Valence charges and the electron-count check** (schema 0.3).
4. **Adapters, offline.** Run the OPTIMADE and OQMD adapters on recorded-shape responses and compare three descriptions of silicon from three sources.
5. **The `numpy_planewave` exporter.** Export an entry, reload the arrays, and look at the plane-wave set and the ionic structure factor.
6. **Round trip.** Attach the export record to the entry and send it through JSON.

Everything runs offline from a checkout and needs no API key. Section 4 has one switch (`LIVE`) that turns on real requests to the Materials Project and Alexandria OPTIMADE servers.

> The OPTIMADE and OQMD cells replay *recorded-shape* documents embedded below, not live responses. What they show is how the adapters convert and what they assume, not new facts about the databases.

In [ ]:
import copy
import json
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import qmatbridge
from qmatbridge.basis import cell_volume, num_plane_waves_from_ecut
from qmatbridge.adapters.optimade import OptimadeAdapterConfig, fetch_entry_from_optimade
from qmatbridge.adapters.oqmd import OQMDAdapterConfig, fetch_entry_from_oqmd
from qmatbridge.io import read_entry_json, write_entry_json
from qmatbridge.registry import get_exporter, list_adapters, list_exporters

# Jupyter starts the kernel in the notebook's folder, so find the repo root by
# walking up from wherever we are instead of assuming the current directory.
start = Path.cwd().resolve()
repo_root = next(
    (
        p
        for p in (start, *start.parents)
        if (p / "pyproject.toml").is_file() and (p / "examples").is_dir()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError(
        f"Could not find the QMatBridge repo above {start}; "
        "open this notebook from inside a checkout."
    )
fixtures_dir = repo_root / "benchmarks" / "fixtures"

LIVE = False  # True: section 4 also queries the Materials Project and Alexandria OPTIMADE servers

print(f"qmatbridge {qmatbridge.__version__}, repo root: {repo_root}")

## 1. Registry

Adapters turn an upstream record into a `QMatEntry`; exporters turn a `QMatEntry` into a downstream format. Both are discovered through the registry, so a third-party package can add one without changing this repository. Listing is lazy: nothing is imported until you ask for a plugin by name.

In [ ]:
for group, plugins in (("adapters", list_adapters()), ("exporters", list_exporters())):
    print(f"{group}:")
    for p in plugins:
        print(f"  {p.name:<18} {'built-in' if p.builtin else 'plugin':<9} {p.target}")

## 2. Structures

Since schema 0.2 an entry can record its atomic positions as fractional coordinates. The helper below rebuilds the cell from the six lattice parameters (standard orientation) and draws it with its atoms. The four materials are chosen to differ in symmetry and in species count.

In [ ]:
KEYS = ["Si", "NaCl", "GaN", "BaTiO3", "LiCoO2", "LiFePO4"]
entries = {k: read_entry_json(fixtures_dir / f"{k}.json") for k in KEYS}


def lattice_vectors(lat):
    # Rows a1, a2, a3 in angstrom, standard orientation (a1 along x, a2 in the xy plane).
    al, be, ga = (np.radians(x) for x in (lat.alpha, lat.beta, lat.gamma))
    a1 = np.array([lat.a, 0.0, 0.0])
    a2 = np.array([lat.b * np.cos(ga), lat.b * np.sin(ga), 0.0])
    cx = lat.c * np.cos(be)
    cy = lat.c * (np.cos(al) - np.cos(be) * np.cos(ga)) / np.sin(ga)
    a3 = np.array([cx, cy, np.sqrt(lat.c**2 - cx**2 - cy**2)])
    return np.array([a1, a2, a3])


def draw_cell(ax, entry, palette):
    s = entry.reference.structure
    vecs = lattice_vectors(s.lattice)
    corners = {c: np.array(c) @ vecs for c in np.ndindex(2, 2, 2)}
    for c, p in corners.items():
        for axis in range(3):
            if c[axis] == 0:
                q = list(c)
                q[axis] = 1
                x, y, z = zip(p, corners[tuple(q)])
                ax.plot(x, y, z, color="0.55", lw=0.9)
    cart = np.array([site.frac_coords for site in s.sites]) @ vecs
    for el in sorted(set(s.species)):
        mask = np.array([site.element == el for site in s.sites])
        ax.scatter(*cart[mask].T, s=90, color=palette[el], edgecolors="k", linewidths=0.5, label=el, depthshade=False)
    ax.set_title(f"{s.formula_reduced}  ({s.num_sites} atoms, {s.lattice.spacegroup_symbol})", fontsize=10)
    ax.set_box_aspect(np.ptp(np.array(list(corners.values())), axis=0))
    ax.set_axis_off()
    ax.legend(loc="lower left", fontsize=8, frameon=False)


elements = sorted({el for e in entries.values() for el in e.reference.structure.species})
cmap = plt.get_cmap("tab20")
palette = {el: cmap(i % 20) for i, el in enumerate(elements)}

fig = plt.figure(figsize=(11, 7))
for i, k in enumerate(KEYS):
    ax = fig.add_subplot(2, 3, i + 1, projection="3d")
    draw_cell(ax, entries[k], palette)
fig.suptitle("Unit cells recorded in the Tier-1 entries (from fractional coordinates)")
fig.tight_layout()
plt.show()

The cells are whatever the source database reports, so a primitive cell of an fcc crystal looks like a rhombohedron rather than a cube. That is also why the fingerprint, which covers the lattice and the positions, identifies a *setup* and not the abstract material (see the quickstart).

## 3. Valence charges and the electron-count check

Schema 0.3 adds `hamiltonian.valence_charges`, the number of valence electrons each pseudopotential contributes. Together with the atomic positions it lets QMatBridge check that the electron count is consistent with the structure, and it gives an exporter the ionic charges $Z_a$.

The committed Tier-1 fixtures were written before that field existed, so their charges are empty. Below we add them for two materials, using the VASP potentials the Materials Project used (Si: 4, Na\_pv: 7, Cl: 7), and let `check_electron_count` confirm the totals. A wrong charge is caught instead of being exported silently.

In [ ]:
CHARGES = {"Si": {"Si": 4.0}, "NaCl": {"Na": 7.0, "Cl": 7.0}}


def with_charges(key):
    e = copy.deepcopy(entries[key])
    e.hamiltonian.valence_charges = dict(CHARGES[key])
    return e


charged = {k: with_charges(k) for k in CHARGES}

print("fixture as stored: charges =", entries["Si"].hamiltonian.valence_charges, "-> total", entries["Si"].valence_electron_total())
for k, e in charged.items():
    total = e.valence_electron_total()
    e.check_electron_count()  # raises ValueError on a mismatch
    print(f"{k:5s} charges {e.hamiltonian.valence_charges}  sum over sites = {total:g}  num_electrons = {e.hamiltonian.num_electrons}  OK")

wrong = with_charges("NaCl")
wrong.hamiltonian.valence_charges["Na"] = 9.0  # Na_sv instead of Na_pv, but num_electrons not updated
try:
    wrong.check_electron_count()
except ValueError as exc:
    print("\nwrong charge is caught:", exc)

print("\ncharges are not part of the fingerprint:", charged["Si"].canonical_hash() == entries["Si"].canonical_hash())

## 4. Adapters, offline

Three descriptions of silicon, from three sources, built by three code paths:

* the **Materials Project fixture** (`mp-149`, live-fetched when the fixture was made),
* an **OPTIMADE** `structures` document in the shape the Materials Project OPTIMADE server returns,
* an **OQMD** `formationenergy` record in the shape the OQMD REST API returns.

The OPTIMADE and OQMD databases publish structures but not the calculation settings, so the adapters take the cutoff (and, for OQMD, a default of 520 eV), the spin treatment and the valence charges from configuration and list them under `hamiltonian.metadata["assumed"]`. The `http_get` hook replays a document instead of making a request.

In [ ]:
OPTIMADE_SI = {
    "id": "mp-149",
    "type": "structures",
    "attributes": {
        "last_modified": "2023-03-11T20:40:53.667000",
        "dimension_types": [1, 1, 1],
        "lattice_vectors": [[3.333573, 0.0, 1.924639], [1.111191, 3.142924, 1.924639], [0.0, 0.0, 3.849278]],
        "cartesian_site_positions": [[3.8891685, 2.7500585, 6.7362365], [0.5555955, 0.3928655, 0.9623195]],
        "species": [{"name": "Si", "chemical_symbols": ["Si"], "concentration": [1.0]}],
        "species_at_sites": ["Si", "Si"],
    },
}
OQMD_SI = {
    "name": "Si",
    "entry_id": 1214579,
    "spacegroup": "Fd-3m",
    "unit_cell": [[0, 2.73, 2.73], [2.73, 0, 2.73], [2.73, 2.73, 0]],
    "sites": ["Si @ 0.875 0.875 0.875", "Si @ 0.125 0.125 0.125"],
    "calculation_label": "standard",
}


def replay(doc, wrap):
    return lambda url, timeout: json.dumps({"data": wrap(doc)}).encode()


optimade_cfg = OptimadeAdapterConfig(
    provider="mp",
    encut_ev=520.0,
    spin_polarized=True,
    functional="PBE",
    http_get=replay(OPTIMADE_SI, lambda d: d),
)
oqmd_cfg = OQMDAdapterConfig(http_get=replay(OQMD_SI, lambda d: [d]))

si = {
    "Materials Project fixture": entries["Si"],
    "OPTIMADE (MP-shaped)": fetch_entry_from_optimade("mp:mp-149", optimade_cfg),
    "OQMD (REST-shaped)": fetch_entry_from_oqmd("oqmd-1214579", oqmd_cfg),
}

header = f"{'source':<26}{'id':<16}{'atoms':>5}{'V (Å^3)':>9}{'electrons':>10}{'plane waves':>12}  fingerprint"
print(header)
for name, e in si.items():
    s, h = e.reference.structure, e.hamiltonian
    print(f"{name:<26}{e.reference.provenance.primary.identifier:<16}{s.num_sites:>5}{cell_volume(s.lattice):>9.2f}"
          f"{h.num_electrons:>10}{h.basis.num_plane_waves:>12}  {e.canonical_hash()[:12]}")

print("\nassumed rather than fetched:")
for name, e in si.items():
    print(f"  {name:<26}{e.hamiltonian.metadata.get('assumed', 'nothing (read from the source calculation)')}")

All three describe the same crystal, yet the fingerprints differ: the sources use different cells and slightly different relaxed volumes, and the provenance differs (the Materials Project fixture and its OPTIMADE twin share the geometry but not the record they came from). Left: the volume per atom of each description. Right: the exact plane-wave count against the cutoff for each cell. Plane waves scale with the cell volume, so a roughly one percent difference in volume moves the curves only by about one percent and the counts can coincide at a given cutoff, as they do at 520 eV here.

In [ ]:
names = list(si)
colors = ["#2a9d8f", "#4c78a8", "#e76f51"]
vol_per_atom = [cell_volume(e.reference.structure.lattice) / e.reference.structure.num_sites for e in si.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.9))
ax1.bar(range(3), vol_per_atom, color=colors)
ax1.set_xticks(range(3), [n.replace(" (", "\n(") for n in names], fontsize=8)
ax1.set_ylabel("Volume per atom (Å³)")
ax1.set_ylim(min(vol_per_atom) * 0.95, max(vol_per_atom) * 1.02)
for i, v in enumerate(vol_per_atom):
    ax1.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
ax1.grid(axis="y", linestyle="--", alpha=0.35)

ecuts = np.arange(100, 901, 25)
for (name, e), color, ls in zip(si.items(), colors, ("-", "--", ":")):
    ax2.plot(ecuts, [num_plane_waves_from_ecut(e.reference.structure.lattice, float(c)) for c in ecuts],
             ls, color=color, lw=2, label=name)
ax2.axvline(520, color="0.5", lw=0.8)
ax2.set_xlabel("Cutoff energy (eV)")
ax2.set_ylabel("Plane waves at Γ")
ax2.legend(fontsize=8)
ax2.grid(linestyle="--", alpha=0.35)
fig.suptitle("Silicon as described by three sources")
fig.tight_layout()
plt.show()

### Live check (optional)

Set `LIVE = True` in the setup cell to fetch two real OPTIMADE entries: the Materials Project's silicon and an Alexandria entry. Alexandria's entry contains Cs, B and C, and Cs has several PAW potentials, so its charges must be given; the adapter refuses to guess. The cell is skipped when `LIVE` is false or the network is unreachable.

In [ ]:
if LIVE:
    cfg = OptimadeAdapterConfig(encut_ev=520.0, timeout_s=60.0, valence_charges={"Cs": 9.0, "B": 3.0, "C": 4.0})
    for ident in ("mp:mp-149", "alexandria-pbe:agm000999956"):
        try:
            e = fetch_entry_from_optimade(ident, cfg)
        except (ConnectionError, ValueError) as exc:
            print(f"{ident}: skipped ({exc})")
            continue
        s = e.reference.structure
        print(f"{ident:<30}{s.formula_reduced:<8}{s.num_sites:>3} atoms  eta={e.hamiltonian.num_electrons:<4} N={e.hamiltonian.basis.num_plane_waves}")
else:
    print("LIVE is False: skipping the network cell.")

## 5. The `numpy_planewave` exporter

The exporter writes what a first-quantized plane-wave Hamiltonian is built from: the G-vectors with their kinetic energies, the ion positions and charges, and the ionic structure factor $S(\mathbf G)=\sum_a Z_a\,e^{-i\mathbf G\cdot\mathbf R_a}$. It refuses an entry that has no positions or whose charges do not match the electron count, which is why section 3 added charges first.

The returned `ExportMetadata` records the framework, format, file and the entry's fingerprint, so an artifact can always be traced back to the entry it came from.

In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="qmatbridge_tour_"))
exporter = get_exporter("numpy_planewave")

exports, arrays = {}, {}
for k, e in charged.items():
    meta = exporter.export(e, path=workdir / f"{k}.npz")
    e.exports.append(meta)
    exports[k] = meta
    with np.load(meta.artifact_path) as f:
        arrays[k] = {name: f[name] for name in f.files}
    n = arrays[k]["g_vectors"].shape[0]
    assert n == e.hamiltonian.basis.num_plane_waves  # the exported set matches the recorded count exactly
    print(f"{k:5s} {meta.format}  N = {n}  atoms = {meta.metadata['num_atoms']}  "
          f"file = {Path(meta.artifact_path).name} ({Path(meta.artifact_path).stat().st_size / 1024:.0f} KiB)")

print("\narrays in the Si file:", ", ".join(f"{n}{arrays['Si'][n].shape}" for n in arrays["Si"] if arrays["Si"][n].ndim))

### The plane-wave set

Left: the G-vectors that fit under the cutoff form a ball in reciprocal space, here coloured by kinetic energy $\hbar^2|\mathbf G|^2/2m$. Right: the number of plane waves with kinetic energy up to $E$ is a staircase that follows $N(E)\propto E^{3/2}$ (dashed, scaled to the value at the cutoff), the same law the quickstart uses to scale the basis with the cutoff.

In [ ]:
fig = plt.figure(figsize=(11, 4.6))
ax3 = fig.add_subplot(1, 2, 1, projection="3d")
g = arrays["Si"]["g_vectors"]
sc = ax3.scatter(*g.T, c=arrays["Si"]["kinetic_energy_ev"], s=6, cmap="viridis")
ax3.set_xlabel("$G_x$ (1/Å)")
ax3.set_ylabel("$G_y$ (1/Å)")
ax3.set_zlabel("$G_z$ (1/Å)")
ax3.set_title("Si: plane waves under the 520 eV cutoff")
ax3.set_box_aspect((1, 1, 1))
fig.colorbar(sc, ax=ax3, shrink=0.65, pad=0.08, label="kinetic energy (eV)")

ax = fig.add_subplot(1, 2, 2)
for k, color in (("Si", "#2a9d8f"), ("NaCl", "#e76f51")):
    ke = arrays[k]["kinetic_energy_ev"]
    ecut = float(arrays[k]["cutoff_energy_ev"])
    ax.step(ke, np.arange(1, ke.size + 1), where="post", color=color, label=f"{k} (exact)")
    grid = np.linspace(0, ecut, 100)
    ax.plot(grid, ke.size * (grid / ecut) ** 1.5, "--", color=color, alpha=0.6)
ax.set_xlabel("kinetic energy $E$ (eV)")
ax.set_ylabel("plane waves with kinetic energy $\\leq E$")
ax.set_title("$N(E)$ and the $E^{3/2}$ law (dashed)")
ax.legend()
ax.grid(linestyle="--", alpha=0.35)
fig.tight_layout()
plt.show()

### The ionic structure factor

$|S(\mathbf G)|$ is what the ions contribute to the potential at each plane wave. Silicon's two atoms sit on the diamond sites, which switches off some reflections entirely (the structure factor vanishes for them); the cell below counts them. The charges used here are set by the pseudopotential choice in section 3, so the NaCl pattern, with equal charges on Na and Cl, is a property of those numbers and not a general feature of the compound.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.9), sharey=False)
for ax, (k, color) in zip(axes, (("Si", "#2a9d8f"), ("NaCl", "#e76f51"))):
    a = arrays[k]
    gnorm = np.linalg.norm(a["g_vectors"], axis=1)
    sf = np.abs(a["ionic_structure_factor"])
    zero = sf < 1e-8 * max(sf.max(), 1.0)
    ax.scatter(gnorm[~zero], sf[~zero], s=7, color=color, label="$|S(G)| > 0$")
    ax.scatter(gnorm[zero], sf[zero], s=7, color="0.6", label="$S(G) = 0$")
    ax.set_xlabel("$|G|$ (1/Å)")
    ax.set_ylabel("$|S(G)|$")
    ax.set_title(f"{k}: {int(zero.sum())} of {sf.size} plane waves have $S(G)=0$")
    ax.grid(linestyle="--", alpha=0.35)
    ax.legend(fontsize=8)
    total = float(a["atom_charges"].sum())
    print(f"{k}: S(0) = {a['ionic_structure_factor'][0].real:g}  (sum of ionic charges = {total:g}); "
          f"zero-structure-factor plane waves: {int(zero.sum())}")
fig.tight_layout()
plt.show()

## 6. Round trip

The export record is part of the entry, so it survives a write and a read. The fingerprint does not change when an export is attached, which is the point: the artifact is *about* the entry and is not part of its identity.

In [ ]:
e = charged["Si"]
path = workdir / "Si_entry.json"
write_entry_json(e, path)
back = read_entry_json(path)

print("exports recorded:", [(x.framework, x.format, Path(x.artifact_path).name) for x in back.exports])
print("round trip equal:", back == e)
print("fingerprint unchanged by the export record:", back.canonical_hash() == entries["Si"].canonical_hash())
print("export metadata carries the entry's fingerprint:", back.exports[0].metadata["entry_hash"] == back.canonical_hash())

## Where to go next

* Fetch your own entries with the Materials Project adapter (`pip install "qmatbridge[mp]"`, `MP_API_KEY`) or with the OPTIMADE adapter against any compliant server (`OptimadeAdapterConfig(base_url=...)`).
* Pass `valence_charges` (and your own cutoff) when an adapter refuses to guess; the exporter then gives you a ready-to-use set of plane-wave arrays.
* Exporters for OpenFermion and for LCU coefficients are on the roadmap.